In [1]:
%load_ext autoreload
%autoreload 2

import json
from pathlib import Path

from src.agent import run_agent
from src.langgraph_agent import run_langgraph_agent

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()



In [2]:

def run_raw_with_trace(user_message: str) -> dict:
    """Wrap raw agent to return same shape as LangGraph version."""
    # If your run_agent already returns dict with tool_calls, adapt this
    # Otherwise, use the run_agent_with_trace helper from Week 2 Day 5
    from src.agent import run_agent_with_trace
    trace = run_agent_with_trace(user_message)
    return {
        "final_answer": trace["final_answer"],
        "tool_calls": trace["tool_calls"],
    }



In [11]:

# Load test set
test_set = json.loads((PROJECT_ROOT / "data" / "test_set.json").read_text())


In [ ]:

# Run both agents on same questions
comparison = []
for q in test_set[:5]:  # first 5 to keep runtime manageable
    print(f"\n[{q['id']}] {q['question'][:60]}...")

    raw_result = run_raw_with_trace(q["question"])
    lg_result = run_langgraph_agent(q["question"], verbose=False)

    raw_tools = [tc["name"] for tc in raw_result["tool_calls"]]
    lg_tools = [tc["name"] for tc in lg_result["tool_calls"]]

    comparison.append({
        "id": q["id"],
        "question": q["question"],
        "raw_tools": raw_tools,
        "lg_tools": lg_tools,
        "raw_answer_len": len(raw_result["final_answer"]),
        "lg_answer_len": len(lg_result["final_answer"]),
        "same_tools": raw_tools == lg_tools,
    })
    print(f"  raw: {raw_tools} | lg: {lg_tools} | match: {raw_tools == lg_tools}")

# Summary
same_count = sum(1 for c in comparison if c["same_tools"])
print(f"\n\n=== Summary ===")
print(f"Same tool sequence: {same_count}/{len(comparison)}")
print(f"Divergences:")
for c in comparison:
    if not c["same_tools"]:
        print(f"  {c['id']}: raw={c['raw_tools']} vs lg={c['lg_tools']}")



[q1] What is retrieval augmented generation according to the book...
  raw: ['search_book'] | lg: ['search_book'] | match: True

[q2] What topics does this book cover?...
  raw: ['list_topics'] | lg: ['list_topics'] | match: True

[q3] Hello, can you introduce yourself?...
  raw: [] | lg: [] | match: True

[q4] What is the Riemann Hypothesis?...
  raw: [] | lg: [] | match: True

[q5] What is retrieval augmented generation and compared to fine-...
  raw: ['search_book', 'search_book'] | lg: ['search_book', 'search_book'] | match: True


=== Summary ===
Same tool sequence: 5/5
Divergences:


ModuleNotFoundError: No module named 'pandas'

In [14]:

# Save for later
import pandas as pd
df = pd.DataFrame(comparison)
df.to_csv(PROJECT_ROOT / "data" / "week3_agent_comparison.csv", index=False)
print(f"\n✓ Saved comparison to data/week3_agent_comparison.csv")


✓ Saved comparison to data/week3_agent_comparison.csv


### Summary:
*what value does LangGraph add if behavior is the same?

LangGraph's value isn't behavior — it's inspectability, composability, 
and the primitives it unlocks.

Even with identical behavior on my test set, LangGraph gives me:
- A graph structure I can trace and diagram
- Nodes I can wrap or embed in larger graphs later
- Conditional edges (Day 5) that would require hand-written routing 
  logic in the raw SDK
- Built-in checkpointing for stateful agents
- Ecosystem tooling (LangSmith, streaming, HITL) I can adopt without 
  rebuilding

For my current 4-tool book agent, none of these matter enough to justify 
the framework overhead. For a fraud-investigation agent with multiple 
sub-agents, they'd matter a lot. Framework value scales with complexity.